# ♟️ Chess Numba — 新舊引擎錦標賽對戰 (Colab Tournament Runner)

本 Notebook 用於在 Google Colab 雲端高並行 CPU 環境下，讓 **新版本引擎 (Candidate / New)** 與 **基準版本引擎 (Baseline / Old)** 進行成對換先（Color-swapped Pairs）錦標賽對抗：

### 🎯 本輪對抗設定 (實驗 E86：SF11 評估核心漏洞修復與全模組 8-Block Texel 重調參驗證)
* **Engine 1 (New / Candidate)**: `chess_engine/classical`
  * ✅ **SF11 評估核心漏洞修復**：
    1. `WeakQueen`: 修正發現攻擊線遮擋子計算，對齊 SF11 `slider_blockers`（消除多重遮擋誤判為弱后之 Bug）。
    2. `MinorBehindPawn`: 對齊 SF11 輕子後方支援兵判定（包含己方與敵方兵）。
    3. `ThreatByKing`: 對齊 SF11 布林單次判定（修正重複計數問題）。
    4. `King-Pawn Endgame Tempo`: 修正王兵殘局 Scale Factor 縮放後遺漏加上 Tempo Bonus 的計分偏差。
    5. `Passed Pawn Advance Path`: 修正雙步推進路徑遮擋與己方王阻擋判定。
  * ✅ **全模組 8-Block Texel 調參 (62 組評估常數更新)**：
    * 涵蓋 Passer dynamics、Threats & Outposts、Mobility、Connected pawns、King safety & Shelter/Storm、PST shapes、Material imbalance & Space、Joint Polish。
    * 於 2,000,000 局面 holdout 驗證損失函數顯著下降 ($z = -72.85$)，PGN 獨立資料集驗證 ($z = -52.37$)。
* **Engine 2 (Old / Baseline)**: `chess_engine/classical_old`（修復與調參前的 Elo 基準版本）
* **時控模式**: 固定節點（400,000 nodes/move）
* **對戰規格**: 1,000 局（500 對開局交換白黑），64MB 轉置表，開局庫 `data/openings.epd`
* **統計模型**: SPRT 與 Elo 95% 置信區間


In [1]:
# =====================================================================
# 步驟 1: 檢視 Colab 雲端硬體規格與 CPU 核心數
# =====================================================================
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!free -h
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.4Gi        47Gi       2.2Mi       2.7Gi        49Gi
Swap:             0B          0B          0B
Python 3.13.18


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
# =====================================================================
# 步驟 3: 嚴格核對新舊引擎代碼與評估常數差異（驗證 Candidate vs Baseline）
# =====================================================================
import sys
from pathlib import Path

# 1) 清理 Python 模組記憶體快取並同步 Snapshot
for m in list(sys.modules.keys()):
    if 'chess_engine' in m or 'tune_search' in m or 'tuner' in m:
        del sys.modules[m]
!python -m tuner.sync_constants_snapshot

# 2) 顯示新舊版本邏輯代碼差異（驗證 evaluation.py, material.py 等修復）
print("=" * 60)
print("🔍 新舊引擎邏輯代碼差異比對 (classical vs classical_old)：")
print("=" * 60)
!python tools/sync_classical_old.py --diff-code

# 3) 顯示新舊版本評估常數差異（驗證 62 組 Texel 調參常數）
print("\n" + "=" * 60)
print("⚖️ 新舊引擎評估常數差異比對 (constants.py)：")
print("=" * 60)
!python tools/sync_classical_old.py --diff-constants

# 4) 快速合約測試 (確保 JIT 與核心介面運作正常)
print("\n" + "=" * 60)
print("🔥 執行單元合約測試：")
print("=" * 60)
!python -m unittest tests/test_search_core_contracts.py


/content
Cloning into 'chess_numba'...
remote: Enumerating objects: 4926, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 4926 (delta 164), reused 161 (delta 120), pack-reused 4694 (from 2)
Receiving objects: 100% (4926/4926), 208.30 MiB | 28.87 MiB/s, done.
Resolving deltas: 100% (3464/3464), done.
/content/chess_numba
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
Already on 'feat-initial-bitboard-setup'
Your branch is up to date with 'origin/feat-initial-bitboard-setup'.
HEAD is now at 75c6cb6 feat(eval): complete full Texel retuning post bugfixes and update constants
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

🎉 環境與相依套件安裝完畢！


In [3]:
# =====================================================================
# 步驟 3: 嚴格核對新舊引擎代碼差異（驗證 Candidate vs Baseline）
# =====================================================================
import sys
from pathlib import Path

# 1) 清理 Python 模組記憶體快取
for m in list(sys.modules.keys()):
    if 'chess_engine' in m or 'tune_search' in m:
        del sys.modules[m]
!python -m tuner.sync_constants_snapshot
# 2) 顯示新舊版本差異（確認 classical vs classical_old 確實包含本次修改）
print("=" * 60)
print("🔍 新舊引擎邏輯代碼差異比對 (classical vs classical_old)：")
print("=" * 60)
!python tools/sync_classical_old.py --diff-code

# 3) 快速合約測試 (確保 JIT 與核心介面運作正常)
print("\n" + "=" * 60)
print("🔥 執行單元合約測試：")
print("=" * 60)
!python -m unittest tests/test_search_core_contracts.py


Unchanged tuner/constants_snapshot.py
🔍 新舊引擎邏輯代碼差異比對 (classical vs classical_old)：

[!] Differences found in evaluation.py:
--- classical_old/evaluation.py
+++ classical/evaluation.py
@@ -30,7 +30,7 @@
     evaluate_pawn_structure, evaluate_pawn_structure_cached,
     evaluate_piece_coordination, evaluate_shelter_aligned
 )
-from chess_engine.classical.material import compute_imbalance
+from chess_engine.classical.material import compute_imbalance, _trunc_div
 
 # Runtime HCE weight indices (must match tune_eval_match.eval_weights)
 _EW_TEMPO = 0
@@ -139,11 +139,6 @@
 
 
 @numba.njit(cache=True, inline="always")
-def _trunc_div(n, d):
-    return n // d if n >= 0 else -((-n) // d)
-
-
-@numba.njit(cache=True, inline="always")
 def _ew_get(ew, use_ew, idx, default):
     if use_ew:
         return ew[idx]
@@ -558,7 +553,7 @@
     black_eg_score = black_eg_shield + b_extra_eg - b_penalty_eg
 
     mg_safety_score = white_mg_score - black_mg_score
-    eg_safety_score = ((white_eg_score -

In [ ]:
# =====================================================================
# 步驟 4: 設定對戰參數並啟動錦標賽
# =====================================================================
import multiprocessing

# 自動偵測可用的 CPU 核心數 (留 1 核給主行程排程)
cpu_count = multiprocessing.cpu_count()
auto_concurrency = max(1, cpu_count - 1)

# --- 參數設定 ---
GAMES = 1000                 # 對局總數 (必須為偶數；400 局可獲穩健 Elo 95% CI)
NODES = 400000              # 每步節點預算 (300k 長時控；快速測試可改為 50000)
TT_MB = 64                  # 每個 Worker 的轉置表大小 (MB)
CONCURRENCY = auto_concurrency  # 平行對局數 (例如 7 或 11)

print("=" * 60)
print(f"♟️ 錦標賽參數配置：")
print(f"  總局數: {GAMES} 局 ({GAMES // 2} 對成對換先開局)" 
      f"  時控: {NODES:,} nodes/move" 
      f"  並行 Worker: {CONCURRENCY}" 
      f"  轉置表: {TT_MB} MB / context")
print("=" * 60)

%cd /content/chess_numba
!python tools/tournament.py --games {GAMES} --nodes {NODES} --concurrency {CONCURRENCY} --tt-mb {TT_MB} --seed 31415926


♟️ 錦標賽參數配置：
  總局數: 1000 局 (500 對成對換先開局)  時控: 400,000 nodes/move  並行 Worker: 7  轉置表: 64 MB / context
/content/chess_numba


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================================
# 步驟 5: 產出統計報告與搜尋剪枝指標診斷
# =====================================================================
%cd /content/chess_numba
!python tournament_analysis/統計數據.py
!python tournament_analysis/parse_search_stats.py

# report_path = Path("tournament_analysis/game_stats_report.txt")
# if report_path.exists():
#     print("\n" + "=" * 60)
#     print("📊 對戰統計結果報告 (Game Stats Report)：")
#     print("=" * 60)
#     print(report_path.read_text(encoding='utf-8'))

# search_report_path = Path("tournament_analysis/search_stats_report.txt")
# if search_report_path.exists():
#     print("\n" + "=" * 60)
#     print("🔬 搜尋樹指標與剪枝診斷報告 (Search Stats Report)：")
#     print("=" * 60)
#     print(search_report_path.read_text(encoding='utf-8'))


In [ ]:
# =====================================================================
# 步驟 6: 視覺化關鍵圖表展示
# =====================================================================
import os
from IPython.display import Image, display

charts = [
    ("累積分數曲線 (Cumulative Score)", "tournament_analysis/cumulative_score.png"),
    ("Elo 置信區間收斂圖 (Elo CI)", "tournament_analysis/elo_ci.png"),
    ("每手搜尋深度 (Depth per Move)", "tournament_analysis/depth_per_move.png"),
    ("同深度節點比例分佈 (Same-Depth Node Ratio)", "tournament_analysis/same_depth_node_ratio.png"),
    ("平均節點數 vs 深度散佈 (Avg Nodes vs Depth)", "tournament_analysis/avg_nodes_vs_depth.png"),
    ("黑白勝負分佈 (WDL by Color)", "tournament_analysis/wdl_by_color.png"),
]

for title, path in charts:
    if os.path.exists(path):
        print(f"\n📈 {title}：")
        display(Image(filename=path))


In [ ]:
# =====================================================================
# 步驟 7: 一鍵打包所有報告、圖表與 PGN 棋譜並下載
# =====================================================================
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"tournament_results_{timestamp}.zip"

!zip -q {zip_filename} tournament_analysis/*.png tournament_analysis/*.txt tournament_analysis/tournament_results.pgn
print(f"📦 已打包完成: {zip_filename}")

try:
    from google.colab import files
    files.download(zip_filename)
    print("⬇️ 已觸發下載！")
except ImportError:
    print(f"💡 本地環境執行，壓縮包位於當前目錄: {zip_filename}")


In [ ]:
from google.colab import runtime

# 執行這行會直接中斷並釋放目前使用的虛擬機器運算資源
runtime.unassign()
